In [16]:
import numpy as np
import time
import torch
import torch.nn.functional as F
import os
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import tenseal as ts

MODEL_ID = "Snowflake/snowflake-arctic-embed-m-v1.5"
FULL_DIM = 768 # native dimensionality of the model
NUM_DOCS = 1000
NUM_QUERIES = 200
TOP_K = 10

device = torch.device("cpu")

In [17]:
def load_corpus_and_queries():
    """Build the doc DB (unique contexts) and query set (questions)."""
    ds = load_dataset("rajpurkar/squad", split="validation")

    # Unique contexts, preserving first-seen order
    contexts, seen = [], set()
    for ex in ds:
        c = ex["context"]
        if c not in seen:
            seen.add(c)
            contexts.append(c)
        if len(contexts) == NUM_DOCS:
            break
    assert len(contexts) == NUM_DOCS

    # Queries: questions whose gold context is in the DB
    ctx_to_id = {c: i for i, c in enumerate(contexts)}
    queries, gold = [], []
    for ex in ds:
        c = ex["context"]
        if c in ctx_to_id:
            queries.append(ex["question"])
            gold.append(ctx_to_id[c])
        if len(queries) == NUM_QUERIES:
            break
    assert len(queries) == NUM_QUERIES
    return contexts, queries, gold


def encode(texts, is_query=False, batch_size=64):
    """Return one embedding per text, L2-normalized, on CPU.

    Arctic models are *asymmetric*: queries get a prompt prefix,
    documents are encoded as-is (this matches the model card).
    """
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

    if is_query:
        texts = ["Represent this sentence for searching relevant passages: " + t
                 for t in texts]

    vecs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tok(
                texts[i:i + batch_size],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(device)
            out = model(**batch)

            # mean pooling over non-padding tokens
            mask = batch["attention_mask"].unsqueeze(-1).float()
            emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
            emb = F.normalize(emb, p=2, dim=1)
            vecs.append(emb)
    return torch.cat(vecs)  # (n, 768), float32, unit norm


def truncate(matryoshka_vecs, dim):
    """Keep the first `dim` components and re-normalize.

    Re-normalization is essential: the prefix has norm < 1 in general,
    and cosine similarity is scale-invariant, so this preserves the
    ranking geometry the model was trained to have.
    """
    return F.normalize(matryoshka_vecs[:, :dim], p=2, dim=1)

# --------------------------------
# Part (c): 1. cosine similarity function
#           2. top_k selection
# --------------------------------

def cosine_similarity(query_vec, doc_matrix):
    """Cosine similarity of one query against all documents.

    Derived from the Euclidian dot product formula:
    A @ B = norm(A) * norm(B) * cos(theta)
    <=>
    cos(theta) = (A @ B) / (norm(A) * norm(B))
    """

    q = np.asarray(query_vec, dtype=np.float64)
    D = np.asarray(doc_matrix, dtype=np.float64)
    dots = D @ q
    qn = np.linalg.norm(q)
    dn = np.linalg.norm(D, axis=1)
    return dots / np.maximum(qn * dn, 1e-12)

def top_k_indices(scores, k):
    """Indices of k largest scores, in descending order"""
    s = np.array(scores, dtype=np.float64, copy=True)
    k = min(k, s.shape[0])
    out = []
    for _ in range(k):
        i = int(np.argmax(s))
        out.append(i)
        s[i] = -np.inf
    return out

def recall_at_k(docs, queries, gold, k=TOP_K):
    """Recall@k: fraction of queries whose gold doc is in the top-k."""
    docs = np.asarray(docs)
    queries = np.asarray(queries)
    hits = 0
    for qi in range(queries.shape[0]):
        sims = cosine_similarity(queries[qi], docs)
        if gold[qi] in top_k_indices(sims, k):
            hits += 1
    return hits / queries.shape[0]

In [18]:
class RetrieverServer:
    """Class that stores the 1,000 documents and their vectors. The client
    may only talk to it through a single search(query_vector, k) call.
    """

    def __init__(self, contexts, embed_dim=256):
        # Embedding the corpus happens here, once.
        self.contexts = list(contexts)
        self.embed_dim = embed_dim
        D_full = encode(self.contexts, is_query=False)     # (1000, 768)

        # Matryoshka truncation to the working dimension, renormalized.
        self.doc_vecs = truncate(D_full, embed_dim)         # (1000, embed_dim)

    def search(self, query_vector, k):
        """Return the ids (and texts) of the k nearest documents.

        query_vector: (embed_dim,) numpy array or float tensor,
        assumed L2-normalized (the client normalizes before sending).
        """

        q = (query_vector.numpy() if isinstance(query_vector, torch.Tensor)
             else np.asarray(query_vector, dtype=np.float64))

        assert q.shape == (self.doc_vecs.shape[1],), \
            "query dimensionality must match the server's embed_dim"

        sims = cosine_similarity(q, self.doc_vecs)
        top_ids = top_k_indices(sims, k)

        return [
            {"id": i, "text": self.contexts[i],
             "score": float(sims[i])}
            for i in top_ids
        ]

def evaluate(server, Q, gold, ks=(1, 5, 10), warmup=10):
    """Recall@{1,5,10} and average per-query latency over the query set."""

    # ---- warmup (untimed) ----
    for q in Q[:warmup]:
        server.search(q, max(ks))

    # ---- timed pass ----
    hits = {k: 0 for k in ks}
    latencies = []
    for qi, qvec in enumerate(Q):
        t0 = time.perf_counter()
        results = server.search(qvec, max(ks)) # single round trip
        latencies.append(time.perf_counter() - t0)

        ids = [r["id"] for r in results]
        for k in ks:
            if gold[qi] in ids[:k]:
                hits[k] += 1

    n = len(Q)
    return ({k: hits[k] / n for k in ks},
            sum(latencies) / n,
            latencies)

In [20]:
contexts, queries, gold = load_corpus_and_queries()

# ---- Part (b): Client/Server split
server = RetrieverServer(contexts, embed_dim=256)

# ---- Client: encode queries, ask the server, measure Recall@10 --
Q_full = encode(queries, is_query=True)     # (200, 768)
Q = truncate(Q_full, 256)                   # (200, 256)

recalls, avg_lat, lat = evaluate(server, Q, gold, ks=(1, 5, 10))

D_full = encode(contexts, is_query=False)   # (1000, 768)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: Snowflake/snowflake-arctic-embed-m-v1.5
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: Snowflake/snowflake-arctic-embed-m-v1.5
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: Snowflake/snowflake-arctic-embed-m-v1.5
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
# ---- Part (c): Evaluation
print(f"\nEvaluation over {len(Q)} queries @ 256 dims, "
        f"{NUM_DOCS} docs")
for k in (1, 5, 10):
    print(f"Recall@{k:<2} = {recalls[k]:.3f}")
print(f"Avg query latency (server round trip): "
        f"{avg_lat * 1000:.2f} ms  (median "
        f"{sorted(lat)[len(lat) // 2] * 1000:.2f} ms)")


Evaluation over 200 queries @ 256 dims, 1000 docs
Recall@1  = 0.480
Recall@5  = 0.900
Recall@10 = 0.965
Avg query latency (server round trip): 1.97 ms  (median 1.92 ms)


In [22]:
# ---- Part (a): Matryoshka dimensions --------
print(f"{'dim':>5} {'Recall@10':>10}")
for d in [64, 128, 256, 768]:
    D, Q = truncate(D_full, d), truncate(Q_full, d)
    r = recall_at_k(D, Q, gold, TOP_K)
    print(f"{d:>5} {r:>10.3f}")

  dim  Recall@10
   64      0.925
  128      0.960
  256      0.965
  768      0.970


### 3.3

In [47]:
class CKKSRetrieverServer:
    """Class which stores the encrypted texts and encrypted and normalized vectors.
    The client interacts with the server by requesting the encrypted scores through the
    scores function and by requesting the ids of a chosen subset (top-k) of documents through
    get_by_ids."""

    def __init__(self, public_context, encrypted_vectors, encrypted_text):
        self.contexts = ts.context_from(public_context)
        self.doc_vecs = [ts.ckks_vector_from(self.contexts, n) for n in encrypted_vectors]
        self.doc_texts = dict(enumerate(encrypted_text))

    def score(self, bytes):
        Q = ts.ckks_vector_from(self.contexts, bytes)
        return ts.CKKSVector.pack_vectors([Q.dot(d) for d in self.doc_vecs]).serialize()

    def get_by_ids(self, ids):
        return [self.doc_texts[i] for i in ids]

In [48]:
def encrypt(aead, doc_id, text, padded_len):
    """
    Encrypts text with AES-GCM.
    The AES-GCM functions work with bytes, so conversion is needed from strings to bytes.
    """
    raw = text.encode("utf-8")
    plaintext = len(raw).to_bytes(4, "big") + raw.ljust(padded_len, b"\x00")
    nonce = os.urandom(12)  # Sets nonce to 12 bytes
    return nonce + aead.encrypt(nonce, plaintext, f"doc:{doc_id}".encode())

def decrypt(aead, doc_id, blob):
    """
    Decrypts with AES-GCM.
    """
    plaintext = aead.decrypt(blob[:12], blob[12:], f"doc:{doc_id}".encode())
    return plaintext[4:4 + int.from_bytes(plaintext[:4], "big")].decode("utf-8")

def encrypted_search(context, aead, server, Q_vector, k):
    """

    """
    encrypted_Q = ts.ckks_vector(context, np.asarray(Q_vector, dtype=np.float64).tolist())
    scores = np.array(ts.ckks_vector_from(context, server.score(encrypted_Q.serialize())).decrypt())
    top_ids = top_k_indices(scores, k)
    return [{"id": i, "text": decrypt(aead, i, b), "score": float(scores[i])}
            for i, b in zip(top_ids, server.get_by_ids(top_ids))]

In [67]:
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree = 8192,
    coeff_mod_bit_sizes = [60,40,40,60])
context.global_scale = 2**40
context.generate_galois_keys()
aead = AESGCM(AESGCM.generate_key(256))
NUM_QUERY_SUBSET = 20 # Need to reduce number of queries, takes too long otherwise; Shouldn't affect the comparisons/results

D_256_truncated = truncate(D_full, 256).double().numpy()  # Renormalizes D at 256 dimensions
padded_len = max(len(c.encode("utf-8")) for c in contexts)
CKKSServer = CKKSRetrieverServer(
    context.serialize(save_secret_key=False),
    [ts.ckks_vector(context, d.tolist()).serialize() for d in D_256_truncated],
    [encrypt(aead, i, c, padded_len) for i, c in enumerate(contexts)])

In [68]:
# 3.2 (c) - Finds recall@1/5/10

Q_256_truncated = truncate(Q_full, 256).double().numpy()  # Renormalizes Q at 256 dimensions

hits = {k: 0 for k in (1, 5, 10)}
for qi in range(NUM_QUERY_SUBSET):
    ids = [r["id"] for r in encrypted_search(context, aead, CKKSServer, Q_256_truncated[qi], 10)]
    for k in hits:
        hits[k] += gold[qi] in ids[:k]

In [70]:
print(f"\nEvaluation over {NUM_QUERY_SUBSET} queries @ 256 dims, "
        f"{NUM_DOCS} docs, with the CKKS protocol")
for k in hits:
    print(f"Recall@{k:<2} = {hits[k] / NUM_QUERY_SUBSET:.3f}")


Evaluation over 20 queries @ 256 dims, 1000 docs, with the CKKS protocol
Recall@1  = 0.750
Recall@5  = 0.900
Recall@10 = 0.900


In [71]:
# Sanity check for plaintext recall@1/5/10 for a lower number of queries
print(f"\nEvaluation over {NUM_QUERY_SUBSET} queries @ 256 dims, "
        f"{NUM_DOCS} docs, without the CKKS protocol")
for k in (1, 5, 10):
    plaintext_recall = recall_at_k(D_256_truncated, Q_256_truncated[:NUM_QUERY_SUBSET], gold[:NUM_QUERY_SUBSET], k)
    print(f"Recall@{k:<2} = {plaintext_recall:.3f}")


Evaluation over 20 queries @ 256 dims, 1000 docs, without the CKKS protocol
Recall@1  = 0.750
Recall@5  = 0.900
Recall@10 = 0.900


In [72]:
# 3.3 (c) - Computes the maximum absolute difference between encrypted and plain-text scores

encrypted_scores = [np.array(ts.ckks_vector_from(context, CKKSServer.score(
                  ts.ckks_vector(context, Q_256_truncated[qi].tolist()).serialize())).decrypt())
              for qi in range(NUM_QUERY_SUBSET)]
plaintext_scores = [cosine_similarity(Q_256_truncated[qi], server.doc_vecs) for qi in range(NUM_QUERY_SUBSET)]

max_diff = max(np.abs(e - p).max() for e, p in zip(encrypted_scores, plaintext_scores))

In [81]:
print(f"Max absolute difference over {NUM_QUERY_SUBSET} queries x {NUM_DOCS} docs: {max_diff:.2e}")

Max absolute difference over 20 queries x 1000 docs: 9.34e-07


In [85]:
# 3.3 (c) - Compares the top-k results to check if they're the same

same = 0
for qi in range(NUM_QUERY_SUBSET):
    encrypted_top = top_k_indices(encrypted_scores[qi], TOP_K)
    plaintext_top = [r["id"] for r in server.search(Q_256_truncated[qi], TOP_K)]
    if encrypted_top == plaintext_top:
        same += 1
    else:
        print(f"Query {qi} mismatch: encrypted {encrypted_top}\n          plaintext {plaintext_top}")
print(f"Identical top-{TOP_K}s: {same}/{NUM_QUERY_SUBSET} queries")

Identical top-10s: 20/20 queries
